<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase29_Synthetic_Procedure_Vocabulary_Gate_Drive_to_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 — Phase 29
## Synthetic Procedure linkage + frozen-vocabulary binding gate
**Google Drive → Google Drive | research reproducibility | public-safe synthetic outputs**

**Prerequisite:** completed Phase 28 and its synthetic patched FHIR bundle. This phase preserves the Phase 28 finding: **8 of 48** original MI-CDM gaps were fixed using synthetic UID fixtures; **40** remained blocked (4 missing Procedure links, 36 unverified vocabulary bindings). Additional PERSON/PROCEDURE/MEASUREMENT vocabulary requirements are tracked separately.

**What this notebook does:**
1. Load the exact verified Phase 28 files from My Drive using hash-based pairing; skip stale Drive listing entries.
2. Create four explicitly synthetic FHIR R4 `Procedure` fixtures and link each `ImagingStudy.procedureReference` to its corresponding procedure. This is **not evidence that an actual MRI procedure was documented**.
3. Run negative tests for absent, cross-person, cross-visit, and duplicate links.
4. Create a human-review **binding template** for a local, frozen Athena `CONCEPT.csv` snapshot. Never invent concept IDs or silently choose lexical matches.
5. Reassess the original 48 gaps, record supplemental OMOP core and event-link gates, create **provisional** target staging files and an explicit provenance sidecar, and save everything back to the same Google Drive folder.

**Not performed:** official HL7 validation; real ADNI processing; real DICOM attestations; official MI-CDM load; actual pyOMOP run; full Arm A completion; Arms B or C; actual cohort-equivalence tests.

FHIR R4 supports `ImagingStudy.procedureReference: Reference(Procedure)`: https://hl7.org/fhir/R4/imagingstudy.html . The generated procedures below are fixture-only; the original Phase 28 file is never overwritten.

In [1]:
# 1 — Connect Drive. For local reproducibility tests only, ICHI2027_DRIVE_ROOT overrides MyDrive.
import os, json, hashlib, copy, csv, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

if os.environ.get('ICHI2027_DRIVE_ROOT'):
    DRIVE_ROOT=Path(os.environ['ICHI2027_DRIVE_ROOT'])
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT=Path('/content/drive/MyDrive')

# Optional: set a precise folder relative to MyDrive, e.g. 'NeuroFHIR_ICHI2027/phase24_outputs'.
# Leave blank to scan MyDrive and skip inaccessible Google Drive entries.
DRIVE_RELATIVE_FOLDER = ''
ROOT=(DRIVE_ROOT / DRIVE_RELATIVE_FOLDER).resolve() if DRIVE_RELATIVE_FOLDER else DRIVE_ROOT
if not ROOT.is_dir(): raise FileNotFoundError(f'Drive folder not found: {ROOT}')
print('Drive search root:', ROOT)

Mounted at /content/drive
Drive search root: /content/drive/MyDrive


In [2]:
# 2 — Discover paired Phase 28 inputs safely, not just the newest filename.
def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()

def candidates(root, glob_name):
    found=[]
    for folder,dirs,files in os.walk(root, onerror=lambda err: None):
        # Skip invisible/trash-like directories and any inaccessible shortcuts.
        dirs[:]=[d for d in dirs if not d.startswith('.') and d not in {'node_modules','.git'}]
        for name in files:
            if Path(name).match(glob_name):
                p=Path(folder)/name
                try:
                    if p.is_file(): found.append(p)
                except (FileNotFoundError, OSError): pass
    def safe_mtime(p):
        try: return p.stat().st_mtime
        except (FileNotFoundError, OSError): return -1
    return sorted(found,key=safe_mtime,reverse=True)

def valid_files_same_folder(folder, pattern):
    for p in candidates(folder,pattern):
        if p.parent==folder: yield p

def get_matching_file(folder, pattern, expected_sha=None):
    for p in valid_files_same_folder(folder,pattern):
        try:
            if expected_sha is None or digest(p)==expected_sha: return p
        except (FileNotFoundError,OSError): continue
    return None

selected=None
errors=[]
for jpath in candidates(ROOT,'ICHI2027_Phase28_SHAREABLE*.json'):
    try:
        s=json.loads(jpath.read_text(encoding='utf-8'))
        if s.get('phase')!='28_official_core_contract_and_synthetic_required_field_disposition': continue
        if not (s.get('gates',{}).get('omop_v55_core_fields_verified_from_commit') and
                s.get('gates',{}).get('synthetic_uid_staging_gap_closure_verified')):
            errors.append(f'{jpath.name}: phase gate not passed');continue
        folder=jpath.parent
        b=get_matching_file(folder,'ICHI2027_Phase28_SYNTHETIC_FHIR_UID_PATCHED*.json',s['synthetic_uid']['patched_bundle_sha256'])
        g=get_matching_file(folder,'ICHI2027_Phase28_GAP_DISPOSITION*.csv')
        c=get_matching_file(folder,'ICHI2027_Phase28_PINNED_OMOP55_CORE_FIELDS*.csv')
        if all((b,g,c)):
            selected=(jpath,s,b,g,c);break
        errors.append(f'{jpath.name}: companion bundle/hash or required CSV absent')
    except (FileNotFoundError,OSError,ValueError,KeyError) as exc:
        errors.append(f'{jpath.name}: {type(exc).__name__}')
if not selected:
    raise FileNotFoundError('Cannot match a valid Phase 28 JSON to its exact patched bundle and CSVs. '+
                            '; '.join(errors[:5]))
P28_JSON,p28,INPUT_BUNDLE,INPUT_GAPS,INPUT_CORE=selected
OUT_DIR=P28_JSON.parent
p28_gaps=pd.read_csv(INPUT_GAPS,dtype={'staging_row_id':str}).fillna('')
core=pd.read_csv(INPUT_CORE).fillna('')
with INPUT_BUNDLE.open(encoding='utf-8') as f: original=json.load(f)
assert len(p28_gaps)==48
assert p28_gaps['resolution_status'].eq('SYNTHETIC_FIXTURE_RESOLVED').sum()==8
assert p28_gaps['resolution_status'].str.startswith('BLOCKED').sum()==40
assert len([e for e in original['entry'] if e['resource']['resourceType']=='Observation'])==16
assert len([e for e in original['entry'] if e['resource']['resourceType']=='ImagingStudy'])==4
assert p28['omop_cdm_v55']['pinned_commit']=='4a910305b2cb74a4fc2b2c34baf44eb0542ff03f'
assert p28['imagewg']['commit']=='8f0a19bb94e941bb335a097bc9851fab672621c7'

# Optional field specification from the exact Phase 27 repaired revision, if still in same folder.
field_path=get_matching_file(OUT_DIR,'ICHI2027_Phase27_REPAIRED_PINNED_TARGET_FIELDS*.csv',
                            p28['phase27_fields_sha256'])
field_inventory=pd.read_csv(field_path) if field_path else pd.DataFrame()
if field_path:
    assert field_inventory.groupby('table').size().to_dict()=={'image_feature':12,'image_occurrence':11}
print('Verified Phase 28 bundle hash:',digest(INPUT_BUNDLE))
print('Pinned ImageWG:',p28['imagewg']['commit'])
print('Pinned OMOP v5.5:',p28['omop_cdm_v55']['pinned_commit'])
print('ImageWG field inventory present:',bool(field_path))
print('Output directory:',OUT_DIR)

Verified Phase 28 bundle hash: 4bdc69c7f382b074faf87dbc1d5540124bd060ae7af5114b23390fd134e3856c
Pinned ImageWG: 8f0a19bb94e941bb335a097bc9851fab672621c7
Pinned OMOP v5.5: 4a910305b2cb74a4fc2b2c34baf44eb0542ff03f
ImageWG field inventory present: True
Output directory: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


### A: Synthetic-only MRI procedure provenance fixture
FHIR R4 `ImagingStudy.procedureReference` directly links a study to a performed `Procedure`. The **original Phase 26–28 synthetic evidence had no Procedure**, so these are newly manufactured research test fixtures and must never be interpreted as attested clinical history. Every new resource carries a synthetic-only tag; the original bundle and hashes remain unchanged.

In [3]:
# 3 — Build deterministic one-per-study synthetic Procedure resources.
SYN='https://example.org/ichi2027/synthetic'
FIXTURE_RECORDED='2026-01-15T00:00:00Z'  # fixed literal to make replay byte-identical
patched=copy.deepcopy(original)
resources=[e['resource'] for e in patched['entry']]
assert not any(x.get('resourceType')=='Procedure' for x in resources), 'Input already contains Procedures; inspect before reapplying.'
patients={x['id']:x for x in resources if x['resourceType']=='Patient'}
studies=sorted((x for x in resources if x['resourceType']=='ImagingStudy'),key=lambda x:x['id'])
assert len(studies)==4 and len(patients)==2

for s in studies:
    sid=s['id']; subj=s.get('subject',{}).get('reference','')
    assert subj.startswith('Patient/') and subj.split('/')[-1] in patients
    assert s.get('started')
    pid='proc-'+sid
    proc={
        'resourceType':'Procedure', 'id':pid, 'status':'completed',
        'meta':{'tag':[{'system':SYN+'/fixture-kind','code':'manufactured-test-fixture'}]},
        'identifier':[{'system':SYN+'/procedure-test-fixture','value':pid}],
        'code':{'coding':[{'system':SYN+'/procedure-code','code':'SYN-BRAIN-MRI',
                            'display':'Synthetic brain MRI acquisition (test only)'}]},
        'subject':{'reference':subj},
        'performedDateTime':s['started']
    }
    s['procedureReference']={'reference':'Procedure/'+pid}
    patched['entry'].append({'resource':proc})
    # Transparent provenance of the fixture construction, NOT a clinical provenance claim.
    patched['entry'].append({'resource':{
        'resourceType':'Provenance','id':'prov-'+pid,'recorded':FIXTURE_RECORDED,
        'target':[{'reference':'Procedure/'+pid}],
        'entity':[{'role':'derivation','what':{'reference':'ImagingStudy/'+sid}}],
        'agent':[{'who':{'reference':'Device/'+('dev-'+subj.split('/')[-1])}}],
        'extension':[{'url':SYN+'/StructureDefinition/synthetic-fixture','valueBoolean':True}]
    }})

procedure_count=sum(e['resource']['resourceType']=='Procedure' for e in patched['entry'])
assert procedure_count==4
print('Created',procedure_count,'explicit synthetic Procedure fixtures. Input bundle unchanged.')

Created 4 explicit synthetic Procedure fixtures. Input bundle unchanged.


In [4]:
# 4 — Link integrity checker + negative controls (project-specific, not HL7 certification).
def link_issues(bundle):
    items=[e['resource'] for e in bundle.get('entry',[]) if 'resource' in e]
    idx={}
    for r in items:
        key=(r.get('resourceType'),r.get('id'))
        if key in idx: return ['DUPLICATE_RESOURCE_ID']
        idx[key]=r
    studs=[r for r in items if r.get('resourceType')=='ImagingStudy']
    seen={}; errs=[]
    for s in studs:
        sid=s['id']; pref=s.get('procedureReference',{}).get('reference','')
        if not pref.startswith('Procedure/'):
            errs.append('MISSING_PROCEDURE_REFERENCE');continue
        pid=pref.split('/',1)[1]; p=idx.get(('Procedure',pid))
        if not p: errs.append('UNRESOLVED_PROCEDURE_REFERENCE');continue
        seen[pid]=seen.get(pid,0)+1
        if p.get('subject',{}).get('reference')!=s.get('subject',{}).get('reference'):
            errs.append('CROSS_PATIENT_PROCEDURE_LINK')
        if p.get('performedDateTime')!=s.get('started'):
            errs.append('CROSS_VISIT_OR_TIME_LINK')
        if not any(t.get('code')=='manufactured-test-fixture' for t in p.get('meta',{}).get('tag',[])):
            errs.append('SYNTHETIC_FIXTURE_LABEL_MISSING')
    if any(v!=1 for v in seen.values()): errs.append('DUPLICATE_PROCEDURE_LINK')
    return sorted(set(errs))

def index_res(bundle,rt):
    return [e['resource'] for e in bundle['entry'] if e['resource']['resourceType']==rt]

checks=[]
def check_fixture(name,b,required):
    observed=set(link_issues(b))
    passed=(observed==set() if not required else set(required).issubset(observed))
    checks.append({'fixture':name,'passed':passed,
                   'expected_codes':';'.join(required),'observed_codes':';'.join(sorted(observed))})
check_fixture('P0_clean',patched,[])
b=copy.deepcopy(patched);index_res(b,'ImagingStudy')[0].pop('procedureReference');check_fixture('P1_missing_reference',b,['MISSING_PROCEDURE_REFERENCE'])
b=copy.deepcopy(patched);index_res(b,'Procedure')[0]['subject']={'reference':'Patient/wrong'};check_fixture('P2_cross_person',b,['CROSS_PATIENT_PROCEDURE_LINK'])
b=copy.deepcopy(patched);index_res(b,'Procedure')[0]['performedDateTime']='2033-01-01T00:00:00Z';check_fixture('P3_cross_time',b,['CROSS_VISIT_OR_TIME_LINK'])
b=copy.deepcopy(patched);ss=index_res(b,'ImagingStudy');ss[1]['procedureReference']=ss[0]['procedureReference'];check_fixture('P4_duplicate_procedure',b,['DUPLICATE_PROCEDURE_LINK'])
b=copy.deepcopy(patched);pr=index_res(b,'Procedure')[0];b['entry']=[e for e in b['entry'] if e['resource'] is not pr];check_fixture('P5_unresolved_procedure',b,['UNRESOLVED_PROCEDURE_REFERENCE'])
fixture_tests=pd.DataFrame(checks)
assert fixture_tests['passed'].all(),fixture_tests.to_string(index=False)
assert not link_issues(patched)
print(f'PASS: {len(fixture_tests)}/{len(fixture_tests)} procedure fixture checks')
display(fixture_tests)

PASS: 6/6 procedure fixture checks


,fixture,passed,expected_codes,observed_codes
0,P0_clean,True,,
1,P1_missing_reference,True,MISSING_PROCEDURE_REFERENCE,MISSING_PROCEDURE_REFERENCE
2,P2_cross_person,True,CROSS_PATIENT_PROCEDURE_LINK,CROSS_PATIENT_PROCEDURE_LINK
3,P3_cross_time,True,CROSS_VISIT_OR_TIME_LINK,CROSS_VISIT_OR_TIME_LINK
4,P4_duplicate_procedure,True,DUPLICATE_PROCEDURE_LINK,CROSS_VISIT_OR_TIME_LINK;DUPLICATE_PROCEDURE_LINK
5,P5_unresolved_procedure,True,UNRESOLVED_PROCEDURE_REFERENCE,UNRESOLVED_PROCEDURE_REFERENCE


### B: Frozen vocabulary, optional reviewed bindings
No concept IDs are guessed. For an evidence-backed run, privately place **`CONCEPT.csv`** (from one chosen, frozen Athena vocabulary download) in a folder named `_PRIVATE_ATHENA` beside the Phase 28 files. The notebook never exports that file. Populate `ICHI2027_Phase29_REVIEWED_BINDINGS.csv` using the generated template, with concept IDs reviewed against **the same snapshot**. Unpopulated/missing or semantically unreviewed bindings stay blocked.

Source-coded values and mapping justification must be checked by a knowledgeable investigator. `CONCEPT.csv` validation proves only snapshot presence, dates, domain, standard-flag, and human-review fields. It **does not itself prove the clinical mapping is correct or that Athena has approved the proposed mapping**.

In [5]:
# 5 — Write reusable human-review binding template; never fill concept IDs automatically.
BINDINGS=[
 ('IO_MODALITY_MR','image_occurrence.modality_concept_id','MR','',True),
 ('IF_HIP_LEFT','image_feature.image_feature_concept_id','SYN-HIP-LEFT','',True),
 ('IF_HIP_RIGHT','image_feature.image_feature_concept_id','SYN-HIP-RIGHT','',True),
 ('IF_ENT_LEFT','image_feature.image_feature_concept_id','SYN-ENT-LEFT','',True),
 ('IF_ENT_RIGHT','image_feature.image_feature_concept_id','SYN-ENT-RIGHT','',True),
 ('IF_TYPE_ALGORITHM','image_feature.image_feature_type_concept_id','SYNTHETIC_ALGORITHM_DERIVED','',True),
 ('PERSON_GENDER_F','person.gender_concept_id','SYNTHETIC_FEMALE','Gender',True),
 ('PERSON_GENDER_M','person.gender_concept_id','SYNTHETIC_MALE','Gender',True),
 ('PERSON_RACE_UNRECORDED','person.race_concept_id','SYNTHETIC_RACE_UNRECORDED','Race',True),
 ('PERSON_ETHNICITY_UNRECORDED','person.ethnicity_concept_id','SYNTHETIC_ETHNICITY_UNRECORDED','Ethnicity',True),
 ('PROCEDURE_MRI','procedure_occurrence.procedure_concept_id','SYN-BRAIN-MRI','Procedure',True),
 ('PROCEDURE_TYPE_SYN','procedure_occurrence.procedure_type_concept_id','SYNTHETIC_FIXTURE_SOURCE','Type Concept',True),
 ('MEAS_HIP_LEFT','measurement.measurement_concept_id','SYN-HIP-LEFT','Measurement',True),
 ('MEAS_HIP_RIGHT','measurement.measurement_concept_id','SYN-HIP-RIGHT','Measurement',True),
 ('MEAS_ENT_LEFT','measurement.measurement_concept_id','SYN-ENT-LEFT','Measurement',True),
 ('MEAS_ENT_RIGHT','measurement.measurement_concept_id','SYN-ENT-RIGHT','Measurement',True),
 ('MEAS_TYPE_SYN','measurement.measurement_type_concept_id','SYNTHETIC_ALGORITHM_DERIVED','Type Concept',True),
 ('UNIT_MM3','measurement.unit_concept_id','mm3','Unit',True),
 ('IF_EVENT_MEAS_ID','image_feature.image_feature_event_field_concept_id','MEASUREMENT.measurement_id','Metadata',False),
]
template=pd.DataFrame([{'binding_key':k,'target_field':tf,'synthetic_source_code':sc,
                        'expected_domain':dom,'requires_standard_concept':std,
                        'concept_id':'','reviewer':'','review_date':'','mapping_evidence':''}
                       for k,tf,sc,dom,std in BINDINGS])
TEMPLATE=OUT_DIR/'ICHI2027_Phase29_BINDING_TEMPLATE.csv'
if not TEMPLATE.exists(): template.to_csv(TEMPLATE,index=False)
REVIEWED=OUT_DIR/'ICHI2027_Phase29_REVIEWED_BINDINGS.csv'
PRIVATE_VOCAB_DIR=OUT_DIR/'_PRIVATE_ATHENA'
CONCEPT=next((PRIVATE_VOCAB_DIR/n for n in ['CONCEPT.csv','concept.csv','CONCEPT.csv.gz'] if (PRIVATE_VOCAB_DIR/n).is_file()),None)
print('Human-review template:',TEMPLATE.name)
print('Reviewed binding file available:',REVIEWED.is_file())
print('Private vocabulary snapshot available:',bool(CONCEPT))

Human-review template: ICHI2027_Phase29_BINDING_TEMPLATE.csv
Reviewed binding file available: False
Private vocabulary snapshot available: False


In [6]:
# 6 — Verify user-reviewed IDs against the local frozen vocabulary snapshot, if supplied.
# Never export raw local vocabulary rows or assume matching concept names imply mapping approval.
validation=[];approved={};vocab_sha=None;reviewed_rows={}
if REVIEWED.is_file():
    review=pd.read_csv(REVIEWED,dtype=str,keep_default_na=False)
    required_cols={'binding_key','concept_id','reviewer','review_date','mapping_evidence'}
    if not required_cols.issubset(review.columns):
        raise ValueError('Reviewed bindings must contain: '+', '.join(sorted(required_cols)))
    if review['binding_key'].duplicated().any(): raise ValueError('Duplicate reviewed binding_key detected.')
    reviewed_rows={r['binding_key']:r for r in review.to_dict('records')}
else:
    review=pd.DataFrame()

chosen=set()
for _,row in template.iterrows():
    r=reviewed_rows.get(row['binding_key'],{})
    cid=str(r.get('concept_id','')).strip()
    if cid:
        if not cid.isdigit() or int(cid)<=0: raise ValueError('Positive integer concept_id required for '+row['binding_key'])
        chosen.add(int(cid))

concepts={}
if CONCEPT and chosen:
    vocab_sha=digest(CONCEPT)
    use=['concept_id','concept_name','domain_id','standard_concept','valid_start_date','valid_end_date','invalid_reason']
    for chunk in pd.read_csv(CONCEPT,sep='\t',compression='infer',dtype=str,
                             chunksize=80000,low_memory=False,usecols=lambda c:c in use):
        if not {'concept_id','domain_id','standard_concept'}.issubset(chunk.columns):
            raise RuntimeError('CONCEPT.csv missing required Athena columns')
        matches=chunk[chunk['concept_id'].isin({str(x) for x in chosen})]
        for cr in matches.to_dict('records'): concepts[int(cr['concept_id'])]=cr
elif CONCEPT:
    vocab_sha=digest(CONCEPT)

asof=datetime.now(timezone.utc).date()
for _,row in template.iterrows():
    k=row['binding_key'];rr=reviewed_rows.get(k,{})
    cid=str(rr.get('concept_id','')).strip()
    status='UNREVIEWED'
    if cid:
        if not CONCEPT: status='BLOCKED_NO_FROZEN_CONCEPT_CSV'
        elif int(cid) not in concepts: status='CONCEPT_NOT_IN_SNAPSHOT'
        else:
            c=concepts[int(cid)]
            try:
                start=datetime.strptime(str(c.get('valid_start_date',''))[:8],'%Y%m%d').date()
                end=datetime.strptime(str(c.get('valid_end_date',''))[:8],'%Y%m%d').date()
                valid=start<=asof<=end
            except (ValueError,TypeError): valid=False
            expected=str(row['expected_domain']).strip()
            domain_ok=not expected or str(c.get('domain_id','')).strip().lower()==expected.lower()
            standard_ok=(not bool(row['requires_standard_concept'])) or str(c.get('standard_concept','')).strip()=='S'
            invalid=str(c.get('invalid_reason','')).strip() not in {'', 'nan','None'}
            reviewer=bool(str(rr.get('reviewer','')).strip())
            reviewed_on=bool(str(rr.get('review_date','')).strip())
            evidence=bool(str(rr.get('mapping_evidence','')).strip())
            if not valid or invalid: status='EXPIRED_OR_INVALID_CONCEPT'
            elif not domain_ok: status='WRONG_DOMAIN'
            elif not standard_ok: status='NOT_STANDARD_CONCEPT'
            elif not (reviewer and reviewed_on and evidence): status='MISSING_EXPLICIT_HUMAN_REVIEW'
            else:
                status='SNAPSHOT_AND_HUMAN_REVIEW_CHECKED'
                approved[k]=int(cid)  # reviewed, NOT proof of authoritative clinical correctness
    validation.append({'binding_key':k,'target_field':row['target_field'],
                       'synthetic_source_code':row['synthetic_source_code'],
                       'concept_id':cid if cid else '', 'validation_status':status,
                       'snapshot_checked':bool(CONCEPT),'human_review_supplied':bool(str(rr.get('reviewer','')).strip())})
binding_audit=pd.DataFrame(validation)
print('Frozen private vocabulary present:',bool(CONCEPT),' | reviewed/snapshot-checked keys:',len(approved),'/',len(BINDINGS))
display(binding_audit[['binding_key','concept_id','validation_status']])

Frozen private vocabulary present: False  | reviewed/snapshot-checked keys: 0 / 19


,binding_key,concept_id,validation_status
0,IO_MODALITY_MR,,UNREVIEWED
1,IF_HIP_LEFT,,UNREVIEWED
2,IF_HIP_RIGHT,,UNREVIEWED
3,IF_ENT_LEFT,,UNREVIEWED
4,IF_ENT_RIGHT,,UNREVIEWED
5,IF_TYPE_ALGORITHM,,UNREVIEWED
6,PERSON_GENDER_F,,UNREVIEWED
7,PERSON_GENDER_M,,UNREVIEWED
8,PERSON_RACE_UNRECORDED,,UNREVIEWED
9,PERSON_ETHNICITY_UNRECORDED,,UNREVIEWED


### C: Gap disposition and provisional target staging
The procedure links here were generated as **synthetic fixtures only**. Even if reviewed concepts are supplied, this phase does not assert that the complete OMOP/MI-CDM pipeline has executed. In particular, source acquisition provenance and real source procedures are not attested.

In [7]:
# 7 — Reconcile the 48 original gap instances without hiding any failures.
def region_side(obs):
    coding=obs.get('bodySite',{}).get('coding',[{}])[0].get('code','')
    match=re.fullmatch(r'SYN-(HIP|ENT)-(LEFT|RIGHT)',coding)
    if not match: raise ValueError('Unrecognized synthetic anatomy/laterality '+coding)
    return match.group(1),match.group(2)

obs=sorted((r for r in index_res(patched,'Observation')),key=lambda x:x['id'])
assert len(obs)==16
source_feature_keys={}
for index,o in enumerate(obs,1):
    region,side=region_side(o)
    source_feature_keys[str(index)]='IF_'+region+'_'+side

# Existing Phase 27 row numbering might differ from Phase 29 sorting; counts can be
# updated from the field types, but concept binding must be keyed from actual source.
# Do NOT claim row-level concept closure if Phase 27 staging order cannot be verified.
disposition=p28_gaps.copy()
mask=disposition['required_field'].eq('procedure_occurrence_id')
assert mask.sum()==4
disposition.loc[mask,'resolution_status']='SYNTHETIC_PROCEDURE_FIXTURE_ONLY'
disposition.loc[mask,'resolution_evidence']='4 synthetic FHIR Procedures linked via ImagingStudy.procedureReference; no real clinical source procedure'
mask=disposition['required_field'].eq('modality_concept_id')
if 'IO_MODALITY_MR' in approved:
    disposition.loc[mask,'resolution_status']='SNAPSHOT_HUMAN_REVIEW_CHECKED_SYNTHETIC'
    disposition.loc[mask,'resolution_evidence']='Local frozen CONCEPT.csv check + explicit review; semantic fitness remains investigator responsibility'
mask=disposition['required_field'].eq('image_feature_type_concept_id')
if 'IF_TYPE_ALGORITHM' in approved:
    disposition.loc[mask,'resolution_status']='SNAPSHOT_HUMAN_REVIEW_CHECKED_SYNTHETIC'
    disposition.loc[mask,'resolution_evidence']='Local frozen CONCEPT.csv check + explicit review; semantic fitness remains investigator responsibility'
# Feature concept gap instances cannot be marked resolved until source row identities
# are reconciled with prior Phase 27 staging, rather than guessed from sorted Obs order.

original_48=disposition['resolution_status'].value_counts().to_dict()
assert len(disposition)==48
assert original_48.get('SYNTHETIC_FIXTURE_RESOLVED',0)==8
assert original_48.get('SYNTHETIC_PROCEDURE_FIXTURE_ONLY',0)==4
print('Original 48 gaps after Phase 29:',original_48)

Original 48 gaps after Phase 29: {'BLOCKED_VOCABULARY_VALIDATION': 36, 'SYNTHETIC_FIXTURE_RESOLVED': 8, 'SYNTHETIC_PROCEDURE_FIXTURE_ONLY': 4}


In [8]:
# 8 — Generate provisional target-shaped rows using verified Phase 27 field inventory.
# Missing vocabulary binding -> null + separately audited required-field gap.
# Stable synthetic source IDs; never real ADNI identifiers.

def getext(o,suffix,key):
    return next((x.get(key) for x in o.get('extension',[]) if x.get('url','').endswith('/'+suffix)),None)

persons=sorted(index_res(patched,'Patient'),key=lambda x:x['id'])
person_ids={p['id']:i+1 for i,p in enumerate(persons)}
occ_ids={s['id']:i+1 for i,s in enumerate(studies)}
synthetic_demographics={persons[0]['id']:(1950,'PERSON_GENDER_F','SYNTHETIC_FEMALE'),
                        persons[1]['id']:(1955,'PERSON_GENDER_M','SYNTHETIC_MALE')}

person_rows=[]
for p in persons:
    yr,gkey,glabel=synthetic_demographics[p['id']]
    person_rows.append({'person_id':person_ids[p['id']], 'gender_concept_id':approved.get(gkey),
      'year_of_birth':yr,'race_concept_id':approved.get('PERSON_RACE_UNRECORDED'),
      'ethnicity_concept_id':approved.get('PERSON_ETHNICITY_UNRECORDED'),
      'person_source_value':p['id'],'gender_source_value':glabel,
      'race_source_value':'SYNTHETIC_UNRECORDED','ethnicity_source_value':'SYNTHETIC_UNRECORDED'})

proc_rows=[];occ_rows=[]
for s in studies:
    sid=s['id'];pid=s['subject']['reference'].split('/')[-1]
    procedure_id=occ_ids[sid];suid=next(
        (x['value'].removeprefix('urn:oid:') for x in s.get('identifier',[])
         if x.get('system')=='urn:dicom:uid'),None)
    series=s['series'][0]['uid']; date=s['started'][:10]
    proc_rows.append({'procedure_occurrence_id':procedure_id,'person_id':person_ids[pid],
      'procedure_concept_id':approved.get('PROCEDURE_MRI'), 'procedure_date':date,
      'procedure_datetime':s['started'],'procedure_type_concept_id':approved.get('PROCEDURE_TYPE_SYN'),
      'procedure_source_value':'SYN-BRAIN-MRI'})
    occ_rows.append({'image_occurrence_id':occ_ids[sid],'person_id':person_ids[pid],
      'procedure_occurrence_id':procedure_id, 'image_occurrence_date':date,
      'image_study_uid':suid,'image_series_uid':series,
      'modality_concept_id':approved.get('IO_MODALITY_MR')})

feat_rows=[];meas_rows=[];lineage_rows=[]
for i,o in enumerate(obs,1):
    sid=o['focus'][0]['reference'].split('/')[-1]
    pid=o['subject']['reference'].split('/')[-1]
    region,side=region_side(o)
    key='IF_'+region+'_'+side
    meas_key='MEAS_'+region+'_'+side
    occ=occ_ids[sid]
    trace_state=getext(o,'trace-state','valueCode')
    device=o['device']['reference']
    dev=next((d for d in index_res(patched,'Device') if 'Device/'+d['id']==device),None)
    alg_version=(dev.get('version') or [{}])[0].get('value') if dev else None
    q=o['valueQuantity']
    meas_rows.append({'measurement_id':i,'person_id':person_ids[pid],
      'measurement_concept_id':approved.get(meas_key),'measurement_date':o['effectiveDateTime'][:10],
      'measurement_datetime':o['effectiveDateTime'], 'measurement_type_concept_id':approved.get('MEAS_TYPE_SYN'),
      'value_as_number':q['value'],'unit_source_value':q.get('unit'),
      'unit_concept_id':approved.get('UNIT_MM3'), 'measurement_source_value':o['code']['text']})
    # Do NOT set event id when event FIELD metadata concept is not validated.
    event_cid=approved.get('IF_EVENT_MEAS_ID')
    feat_rows.append({'image_feature_id':i,'person_id':person_ids[pid],
      'image_occurrence_id':occ,'image_feature_concept_id':approved.get(key),
      'image_feature_type_concept_id':approved.get('IF_TYPE_ALGORITHM'),
      'image_feature_event_field_concept_id':event_cid,
      'image_feature_event_id':i if event_cid is not None else None,
      'alg_system':(dev.get('deviceName') or [{}])[0].get('name') if dev else None,
      'alg_datetime':o['effectiveDateTime']})
    lineage_rows.append({'source_observation_id':o['id'],'source_study_id':sid,
      'synthetic_fhir_procedure_id':'proc-'+sid,'synthetic_procedure_occurrence_id':occ,
      'image_occurrence_id':occ,'image_feature_id':i,'measurement_id':i,
      'synthetic_patient_key':pid, 'anatomy':region,'laterality':side,
      'baseline_or_followup':getext(o,'timepoint-label','valueString'),
      'value':q['value'],'unit':q.get('unit'),
      'algorithm_version':alg_version,'trace_state':trace_state,
      'trace_rule':getext(o,'trace-rule-version','valueString'),
      'event_link_status':'REVIEWED_METADATA_LINK' if event_cid is not None else 'SIDECAR_ONLY_UNVALIDATED_EVENT_FIELD'})

stage={'person':pd.DataFrame(person_rows), 'procedure_occurrence':pd.DataFrame(proc_rows),
       'image_occurrence':pd.DataFrame(occ_rows),'image_feature':pd.DataFrame(feat_rows),
       'measurement':pd.DataFrame(meas_rows)}
lineage=pd.DataFrame(lineage_rows)
assert [len(stage[t]) for t in ['person','procedure_occurrence','image_occurrence','image_feature','measurement']]==[2,4,4,16,16]
assert set(stage['image_feature']['image_occurrence_id']).issubset(set(stage['image_occurrence']['image_occurrence_id']))
assert set(stage['image_occurrence']['procedure_occurrence_id']).issubset(set(stage['procedure_occurrence']['procedure_occurrence_id']))
assert set(lineage['source_observation_id'])=={o['id'] for o in obs}

# If present, do not allow any invented target columns; restrict to pinned ImageWG/OMOP inventories.
for table,frame in stage.items():
    inventory=field_inventory[field_inventory['table'].eq(table)] if not field_inventory.empty and table.startswith('image_') else core[core['table'].eq(table)]
    if not inventory.empty:
        allowed=set(inventory['field'].astype(str))
        unexpected=set(frame.columns)-allowed
        assert not unexpected,(table,unexpected)
        stage[table]=frame.reindex(columns=inventory['field'].tolist())
print('Provisional synthetic stage counts:',{k:len(v) for k,v in stage.items()})
print('MI-CDM field inventory verified for stage:',not field_inventory.empty)

Provisional synthetic stage counts: {'person': 2, 'procedure_occurrence': 4, 'image_occurrence': 4, 'image_feature': 16, 'measurement': 16}
MI-CDM field inventory verified for stage: True


In [9]:
# 9 — Explicitly audit stage mandatory gaps and additional conditional event-link gate.
required_rows=[]
for table,frame in stage.items():
    inventory=field_inventory[field_inventory['table'].eq(table)] if table.startswith('image_') and not field_inventory.empty else core[core['table'].eq(table)]
    if inventory.empty:
        required_rows.append({'table':table,'field':'PINNED_FIELD_INVENTORY_MISSING',
                              'missing_rows':len(frame),'status':'BLOCKED_SCHEMA_INVENTORY_MISSING'})
        continue
    for _,defn in inventory.iterrows():
        if str(defn['required']).strip().lower() not in {'true','1'}:continue
        name=str(defn['field'])
        if name not in frame.columns:
            count=len(frame)
        else:
            count=int(frame[name].isna().sum())
        required_rows.append({'table':table,'field':name,'missing_rows':count,
                              'status':'FILLED_SYNTHETIC_STAGE' if count==0 else 'REQUIRED_UNRESOLVED'})
required_audit=pd.DataFrame(required_rows)
conditional_status=('REVIEWED_EVENT_FIELD_LINK_TEST_ONLY' if 'IF_EVENT_MEAS_ID' in approved else
                    'UNVALIDATED_METADATA_FIELD_KEEP_SIDECAR_LINK')
assert stage['image_feature']['image_feature_event_id'].notna().sum()==(16 if 'IF_EVENT_MEAS_ID' in approved else 0)
assert stage['image_occurrence']['image_study_uid'].notna().all()
assert stage['image_occurrence']['image_series_uid'].notna().all()
print('Required stage fields still missing:',required_audit.loc[required_audit['missing_rows']>0,['table','field','missing_rows']].to_string(index=False))
print('IMAGE_FEATURE event field:',conditional_status)

Required stage fields still missing:                table                         field  missing_rows
              person             gender_concept_id             2
              person               race_concept_id             2
              person          ethnicity_concept_id             2
procedure_occurrence          procedure_concept_id             4
procedure_occurrence     procedure_type_concept_id             4
    image_occurrence           modality_concept_id             4
       image_feature      image_feature_concept_id            16
       image_feature image_feature_type_concept_id            16
         measurement        measurement_concept_id            16
         measurement   measurement_type_concept_id            16
IMAGE_FEATURE event field: UNVALIDATED_METADATA_FIELD_KEEP_SIDECAR_LINK


In [10]:
# 10 — Drive-to-Drive exports. Raw Athena vocabulary is never copied/exported.
OUTPUTS={
 'shareable':OUT_DIR/'ICHI2027_Phase29_SHAREABLE.json',
 'bundle':OUT_DIR/'ICHI2027_Phase29_SYNTHETIC_FHIR_WITH_PROCEDURES.json',
 'procedure_tests':OUT_DIR/'ICHI2027_Phase29_PROCEDURE_LINK_TESTS.csv',
 'gap_disposition':OUT_DIR/'ICHI2027_Phase29_GAP_DISPOSITION.csv',
 'binding_audit':OUT_DIR/'ICHI2027_Phase29_BINDING_AUDIT.csv',
 'required_audit':OUT_DIR/'ICHI2027_Phase29_TARGET_REQUIRED_FIELD_AUDIT.csv',
 'lineage':OUT_DIR/'ICHI2027_Phase29_SYNTHETIC_LINEAGE_SIDECAR.csv',
}
for table in stage:
    OUTPUTS[table]=OUT_DIR/('ICHI2027_Phase29_STAGE_'+table.upper()+'.csv')
with OUTPUTS['bundle'].open('w',encoding='utf-8') as f: json.dump(patched,f,indent=2,ensure_ascii=False)
fixture_tests.to_csv(OUTPUTS['procedure_tests'],index=False)
disposition.to_csv(OUTPUTS['gap_disposition'],index=False)
binding_audit.to_csv(OUTPUTS['binding_audit'],index=False)
required_audit.to_csv(OUTPUTS['required_audit'],index=False)
lineage.to_csv(OUTPUTS['lineage'],index=False)
for t,frame in stage.items():frame.to_csv(OUTPUTS[t],index=False)

remaining_original=int(disposition['resolution_status'].str.startswith('BLOCKED').sum())
all_micdm_requirements_met=bool(
    not field_inventory.empty and
    required_audit.loc[required_audit['table'].isin(['image_feature','image_occurrence']),'missing_rows'].sum()==0
)
all_core_requirements_met=bool(required_audit.loc[required_audit['table'].isin(['person','procedure_occurrence','measurement']),'missing_rows'].sum()==0)
all_bindings_reviewed=bool(len(approved)==len(BINDINGS))
summary={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'29_synthetic_procedure_fixture_and_frozen_vocabulary_gate',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Public synthetic-only outputs; private local Athena vocabulary not exported; no real ADNI data',
 'phase28_sha256':digest(P28_JSON),
 'phase28_patched_bundle_sha256':digest(INPUT_BUNDLE),
 'pinned_imagewg_commit':p28['imagewg']['commit'],
 'pinned_omop55_commit':p28['omop_cdm_v55']['pinned_commit'],
 'synthetic_fixture':{
   'newly_manufactured_procedures':4,'fhir_procedure_reference_links':4,
   'negative_and_clean_test_cases':len(fixture_tests),
   'tests_passing':int(fixture_tests['passed'].sum()),
   'patched_bundle_sha256':digest(OUTPUTS['bundle']),
   'real_source_procedure_attested':False,
 },
 'original_48_gap_disposition':{
   'original_count':48,'synthetic_uid_resolved':8,
   'newly_synthetic_procedure_only_resolved':4,
   'remaining_blocked':remaining_original,
   'status_counts':{str(k):int(v) for k,v in disposition['resolution_status'].value_counts().items()},
   'image_feature_row_order_reconciled_to_phase27':False,
 },
 'vocabulary':{
   'local_frozen_CONCEPT_csv_provided':bool(CONCEPT),
   'local_CONCEPT_sha256':vocab_sha,
   'reviewed_binding_file_provided':REVIEWED.is_file(),
   'reviewed_snapshot_checked_binding_count':len(approved),
   'binding_template_count':len(BINDINGS),
   'Athena_approved_mappings_claimed':False,
 },
 'provisional_stage_row_counts':{t:len(df) for t,df in stage.items()},
 'target_required_missing_instances':int(required_audit['missing_rows'].sum()),
 'event_field_status':conditional_status,
 'gates':{
   'phase28_verified':True,
   'pinned_source_hashes_reconciled':True,
   'synthetic_FHIR_Procedure_fixture_created':True,
   'synthetic_procedure_link_negative_controls_passed':True,
   'real_source_linked_procedure_attested':False,
   'real_DICOM_UID_semantics_attested':False,
   'exact_phase27_imagewg_inventory_read_back':bool(field_path),
   'frozen_vocab_snapshot_provided':bool(CONCEPT),
   'all_proposed_concepts_reviewed_against_snapshot':all_bindings_reviewed,
   'all_MiCDM_required_fields_filled_synthetic_stage':all_micdm_requirements_met,
   'all_core_required_fields_filled_synthetic_stage':all_core_requirements_met,
   'synthetic_faithful_armA_mapping_completed':False,
   'pyOMOP_ETL_executed':False,
   'official_MICDM_load_completed':False,
   'real_MRI_rows_processed':0,
   'true_four_arm_comparison_completed':False,
   'OMOP_target_cohort_equivalence_tested':False,
 },
 'arm_registry':{'A':'PROVISIONAL_SYNTHETIC_STAGING_NOT_ETL_VALIDATED','B':'NOT_EXECUTED',
                 'C':'ORIGINAL_GENERIC_TFL_NONPORTABLE_NOT_EXECUTED',
                 'D':'SYNTHETIC_CONTRACT_PROTOTYPE_ONLY'},
 'next_stage_gate':('Provide a frozen local Athena CONCEPT.csv and reviewed bindings; independently adjudicate '
   'each anatomical/measurement/procedure concept, especially IMAGE_FEATURE event-field metadata; '
   'confirm source row-to-feature ordering. Only then design a separate synthetic mapping/SQL cardinality '
   'and cohort-equivalence test. Real imaging work also requires genuine procedure provenance and actual DICOM UIDs.'),
}
with OUTPUTS['shareable'].open('w',encoding='utf-8') as f:json.dump(summary,f,indent=2)

# Read back saved outputs.
with OUTPUTS['shareable'].open(encoding='utf-8') as f:readback=json.load(f)
assert readback['phase']=='29_synthetic_procedure_fixture_and_frozen_vocabulary_gate'
assert readback['synthetic_fixture']['tests_passing']==6
assert digest(OUTPUTS['bundle'])==readback['synthetic_fixture']['patched_bundle_sha256']
assert len(pd.read_csv(OUTPUTS['gap_disposition']))==48
assert sum(1 for x in json.loads(OUTPUTS['bundle'].read_text())['entry']
           if x['resource']['resourceType']=='Procedure')==4
print('PHASE 29 SYNTHETIC PROCEDURE GATE: PASS')
print('Original 48 gaps: 8 UIDs synthetic-only + 4 Procedures synthetic-only +',remaining_original,'blocked')
print('Remaining blockers are not suppressed by these fixture results.')
print('Saved directly to Google Drive:')
for p in OUTPUTS.values():print(' -',p.name)
print(' -',TEMPLATE.name)

PHASE 29 SYNTHETIC PROCEDURE GATE: PASS
Original 48 gaps: 8 UIDs synthetic-only + 4 Procedures synthetic-only + 36 blocked
Remaining blockers are not suppressed by these fixture results.
Saved directly to Google Drive:
 - ICHI2027_Phase29_SHAREABLE.json
 - ICHI2027_Phase29_SYNTHETIC_FHIR_WITH_PROCEDURES.json
 - ICHI2027_Phase29_PROCEDURE_LINK_TESTS.csv
 - ICHI2027_Phase29_GAP_DISPOSITION.csv
 - ICHI2027_Phase29_BINDING_AUDIT.csv
 - ICHI2027_Phase29_TARGET_REQUIRED_FIELD_AUDIT.csv
 - ICHI2027_Phase29_SYNTHETIC_LINEAGE_SIDECAR.csv
 - ICHI2027_Phase29_STAGE_PERSON.csv
 - ICHI2027_Phase29_STAGE_PROCEDURE_OCCURRENCE.csv
 - ICHI2027_Phase29_STAGE_IMAGE_OCCURRENCE.csv
 - ICHI2027_Phase29_STAGE_IMAGE_FEATURE.csv
 - ICHI2027_Phase29_STAGE_MEASUREMENT.csv
 - ICHI2027_Phase29_BINDING_TEMPLATE.csv


## What to upload here after running
Please upload **`ICHI2027_Phase29_SHAREABLE.json`**, **`ICHI2027_Phase29_GAP_DISPOSITION.csv`**, **`ICHI2027_Phase29_TARGET_REQUIRED_FIELD_AUDIT.csv`**, and the **`ICHI2027_Phase29_BINDING_AUDIT.csv`**. The synthetic FHIR Procedure bundle and provisional staging files may be published in the repository, but **never publish private Athena vocabulary files or real ADNI/DICOM material**. If bindings are missing, use the generated template privately to organize the next vocabulary-validation pass.